# Clinical Strand — Preprocessing & Split

**MultimodalAI‧26 · Clinical Demo**

---

Preprocess the tabular data and create stratified train/val/test splits.
The processed CSVs are the inputs to `02_train_models.ipynb`.

**Prerequisite:** `data/raw/tabular.csv` must exist — it is pre-included in the repository.

**Sections**
1. Setup & Load Data
2. Imputation
3. Train / Val / Test Split
4. Verify Split
5. Save Processed CSVs

---
## Section 1 — Setup & Load Data

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / 'data').exists() and (candidate / 'models').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not find clinical_strand root.')

RAW_DATA = ROOT / 'data' / 'raw' / 'tabular.csv'
PROC_DIR = ROOT / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'major_complication_30d'
print(f'ROOT: {ROOT}')

In [ ]:
df = pd.read_csv(RAW_DATA)
print(f'Shape: {df.shape}')
print(f'Complication rate: {df[TARGET].mean():.1%}')
print(f'Has notes:         {df["has_notes"].mean():.1%}')
print(f'Blood loss MNAR:   {df["blood_loss_missing"].mean():.1%}')
print()
print('Missing values (columns with any NaN):')
miss = df.isnull().sum()
print(miss[miss > 0].to_string())

**Expected output — load:**

```
Shape: (500, 37)
Complication rate: 25.4%
Has notes:         68.8%
Blood loss MNAR:   33.8%

Missing values (columns with any NaN):
blood_loss_ml        169
note_risk_score      156
```

Only `blood_loss_ml` and `note_risk_score` have missing values — both intentionally MNAR. All other columns are complete.

In [ ]:
print('Complication rate by surgery type:')
print(df.groupby('surgery_type')[TARGET].agg(['mean', 'count']).round(3).to_string())
print()
print('Complication rate by notes availability:')
print(df.groupby('has_notes')[TARGET].agg(['mean', 'count']).round(3).to_string())

**Expected output — subgroup breakdown:**

```
Complication rate by surgery type:
               mean  count
cardiac        0.616    112
vascular       0.258     93
abdominal      0.161    155
orthopaedic    0.064    140

Complication rate by notes availability:
           mean  count
0 (absent) 0.385    156
1 (present)0.195    344
```

Cardiac has a 62% complication rate — nearly six times orthopaedic. This imbalance must be preserved in the splits via stratification.

---
## Section 2 — Imputation

Model classes handle `note_risk_score` and `blood_loss_imputed` internally.
For remaining NaN values in numeric columns, apply median imputation.

> **Note:** `blood_loss_ml` NaNs are intentional MNAR — do not impute here.
> The models use `blood_loss_imputed` (pre-computed) and `blood_loss_missing` (flag).
> `note_risk_score` is also left as-is — each model handles it internally.

In [ ]:
numeric_cols = [
    'hr_mean', 'hr_std', 'rr_mean', 'rr_std',
    'spo2_mean', 'spo2_min', 'sbp_mean', 'temp_mean',
    'icu_lactate', 'icu_creatinine', 'icu_wbc', 'icu_bilirubin',
    'preop_creatinine', 'preop_wbc', 'preop_lactate',
]

to_impute = [c for c in numeric_cols if c in df.columns and df[c].isnull().sum() > 0]
for col in to_impute:
    df[col] = df[col].fillna(df[col].median())

if to_impute:
    print(f'Applied median imputation to: {to_impute}')
else:
    print('No NaN in numeric feature columns — no imputation needed.')

skip_cols = ['note_risk_score', 'blood_loss_ml']
remaining = df.drop(columns=[c for c in skip_cols if c in df.columns]).isnull().sum().sum()
print(f'Remaining NaN (excl. MNAR columns): {remaining}')

**Expected output — imputation:**

```
No NaN in numeric feature columns — no imputation needed.
Remaining NaN (excl. MNAR columns): 0
```

All numeric feature columns are already complete. The two intentional MNAR columns (`blood_loss_ml` and `note_risk_score`) are excluded from this count. If you see imputation applied to other columns, check your dataset for unexpected missingness.

---
## Section 3 — Train / Val / Test Split

Stratified split on `major_complication_30d` to preserve class balance across folds.
For 500 patients, 70/15/15 gives roughly 350 train, 75 val, 75 test.

> **Leakage note:** imputation statistics are computed on the full dataset here for
> simplicity. A stricter approach would compute on train only then apply to val/test.
> For 500 patients the difference is negligible, but the principle matters for production.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

df_train, df_temp = train_test_split(
    df, test_size=1 - TRAIN_FRAC, stratify=df[TARGET], random_state=SEED
)
val_share = VAL_FRAC / (1 - TRAIN_FRAC)
df_val, df_test = train_test_split(
    df_temp, test_size=1 - val_share, stratify=df_temp[TARGET], random_state=SEED
)

for split, name in [(df_train, 'train'), (df_val, 'val'), (df_test, 'test')]:
    print(f'{name:6s}: n={len(split):4d}  '
          f'complication={split[TARGET].mean():.1%}  '
          f'notes={split["has_notes"].mean():.1%}')

**Expected output — split sizes:**

```
train : n= 350  complication=25.4%  notes=70.3%
val   : n=  75  complication=25.3%  notes=72.0%
test  : n=  75  complication=25.3%  notes=58.7%
```

All three splits have a complication rate near 25% — stratification is working. Notes availability is slightly lower in the test set, which is fine and reflects natural sampling variation.

---
## Section 4 — Verify Split

In [ ]:
print('Split verification (key subgroup rates must be consistent across splits):')
for name, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    age_b75   = (split['age'] >= 75).mean()
    bl_mnar   = split['blood_loss_missing'].mean()
    print(f'  {name:6s}: complication={split[TARGET].mean():.1%}  '
          f'age75+={age_b75:.1%}  '
          f'notes={split["has_notes"].mean():.1%}  '
          f'blood_loss_mnar={bl_mnar:.1%}')

**Expected output — split verification:**

```
  train : complication=25.4%  age75+=21.7%  notes=70.3%  blood_loss_mnar=34.0%
  val   : complication=25.3%  age75+=17.3%  notes=72.0%  blood_loss_mnar=40.0%
  test  : complication=25.3%  age75+=10.7%  notes=58.7%  blood_loss_mnar=26.7%
```

Complication rates are consistent across all three splits (all ~25%) — the stratified split preserved class balance. Age 75+ and blood-loss MNAR proportions will vary slightly due to the small val/test sizes (75 patients each).

---
## Section 5 — Save Processed CSVs

In [ ]:
df_train.to_csv(PROC_DIR / 'train.csv', index=False)
df_val.to_csv(  PROC_DIR / 'val.csv',   index=False)
df_test.to_csv( PROC_DIR / 'test.csv',  index=False)
print(f'Saved to {PROC_DIR}:')
print(f'  train.csv ({len(df_train)} rows)')
print(f'  val.csv   ({len(df_val)} rows)')
print(f'  test.csv  ({len(df_test)} rows)')
print()
print('Next: run 02_train_models.ipynb')

**Expected output — save:**

```
Saved to .../data/processed:
  train.csv (350 rows)
  val.csv   (75 rows)
  test.csv  (75 rows)

Next: run 02_train_models.ipynb
```

Three files are now in `data/processed/`. Run `02_train_models.ipynb` next.